In [ ]:
# ================================================================
# Stylometric and Statistical Hybrid Detector
# ================================================================
# feature set with:
#   - POS tag distribution (spaCy)
#   - Dependency tree depth (avg + max)
#   - Function word frequency profiles
#   - Punctuation entropy
#   - Per-sentence perplexity mean + variance (GPT-2 small)
#   - Readability indices (Flesch-Kincaid, Gunning Fog)
#
# Classifiers: Logistic Regression, Random Forest, XGBoost
# Primary value: explicit feature importances → which linguistic
# properties distinguish human from AI text across domains.
# ================================================================

# ── Cell 1: Install & Imports ──────────────────────────────────
!pip install -q spacy xgboost shap textstat transformers accelerate
!python -m spacy download en_core_web_sm -q

import os, json, pickle
import pandas as pd
import numpy as np
import torch
import spacy
import textstat
import re, math
from collections import Counter
from scipy import stats as scipy_stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    accuracy_score, roc_curve, classification_report,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from tqdm import tqdm
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nlp = spacy.load("en_core_web_sm", disable=["ner"])  # keep tagger + parser

RESULTS_DIR = "./results/stylometric"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Device : {device}")
print("✅ All imports loaded")

# ── Cell 2: Sentence Perplexity Engine (GPT-2 Small) ──────────
print("\nLoading GPT-2 small for sentence-level perplexity ...")
ppl_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
ppl_model     = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ppl_model.eval()
ppl_tokenizer.pad_token = ppl_tokenizer.eos_token
print("✅ GPT-2 small loaded")

def sentence_perplexity(sentence: str, max_len: int = 256) -> float:
    """Compute GPT-2 perplexity for a single sentence."""
    enc = ppl_tokenizer(sentence, return_tensors="pt",
                        max_length=max_len, truncation=True)
    input_ids = enc["input_ids"].to(device)
    if input_ids.shape[1] < 2:
        return np.nan
    with torch.no_grad():
        loss = ppl_model(input_ids, labels=input_ids).loss
    return float(torch.exp(loss).item())


# ── Cell 3: Function Word Lists ────────────────────────────────
FUNCTION_WORDS = set([
    "the", "a", "an", "of", "in", "on", "at", "to", "for",
    "with", "by", "from", "as", "is", "are", "was", "were",
    "be", "been", "being", "have", "has", "had", "do", "does",
    "did", "will", "would", "could", "should", "may", "might",
    "shall", "can", "that", "this", "these", "those", "which",
    "and", "but", "or", "nor", "so", "yet", "both", "either",
    "not", "no", "nor", "rather", "quite",
])

HEDGING_WORDS = [
    "maybe","perhaps","possibly","probably","might","could",
    "seem","appear","approximately","around","roughly","likely",
    "suggests","indicates","implies","arguably",
]
CERTAINTY_WORDS = [
    "definitely","certainly","obviously","clearly","always","never",
    "absolutely","undoubtedly","evidently","plainly",
]
CONNECTOR_WORDS = [
    "however","therefore","moreover","furthermore","additionally",
    "consequently","nevertheless","nonetheless","meanwhile","thus",
    "hence","whereby","thereby","accordingly","subsequently",
]
AI_HEDGE_PHRASES = [
    "it is worth noting","it is important to","as an ai","note that",
    "in summary","in conclusion","to summarize","first,","second,",
    "third,","finally,","overall,","in general,","generally speaking",
]

# ── Cell 4: Feature Extractor (FULL SUITE) ────────────────────

class FullStylometricExtractor:
    """
    Extended feature set combining Stage 2A features with
    POS, syntax, function words, readability, and sentence-level PPL.
    """

    def __init__(self, use_sentence_ppl: bool = True):
        self.use_sentence_ppl = use_sentence_ppl
        self.feature_names_   = []

    def extract_batch(self, texts):
        rows = []
        for text in tqdm(texts, desc="Extracting features", leave=False):
            rows.append(self._extract_single(str(text)))
        df = pd.DataFrame(rows)
        self.feature_names_ = df.columns.tolist()
        return df

    def _extract_single(self, text: str) -> dict:
        words     = text.split()
        sentences = re.split(r"[.!?]+", text)
        sentences = [s.strip() for s in sentences if s.strip()]
        f = {}

        # ─── FEATURES (preserved) ─────────────────────
        f["word_count"]          = len(words)
        f["char_count"]          = len(text)
        f["sentence_count"]      = len(sentences)
        f["avg_word_len"]        = np.mean([len(w) for w in words]) if words else 0
        f["avg_sentence_len"]    = len(words) / len(sentences) if sentences else 0
        f["type_token_ratio"]    = len(set(words)) / len(words) if words else 0
        word_freq = Counter(words)
        f["hapax_ratio"]         = sum(1 for c in word_freq.values() if c==1) / len(words) if words else 0
        f["comma_density"]       = text.count(",") / len(words) if words else 0
        f["period_density"]      = text.count(".") / len(words) if words else 0
        f["question_mark_ratio"] = text.count("?") / len(sentences) if sentences else 0
        f["exclamation_ratio"]   = text.count("!") / len(sentences) if sentences else 0
        bigrams  = list(zip(words[:-1], words[1:]))
        trigrams = list(zip(words[:-2], words[1:-1], words[2:]))
        f["bigram_repetition"]   = 1 - (len(set(bigrams))/len(bigrams))  if bigrams  else 0
        f["trigram_repetition"]  = 1 - (len(set(trigrams))/len(trigrams)) if trigrams else 0
        wprobs = np.array(list(word_freq.values())) / len(words)
        f["word_entropy"]        = scipy_stats.entropy(wprobs)
        sent_lens = [len(s.split()) for s in sentences]
        f["sentence_len_variance"] = np.var(sent_lens)  if sent_lens else 0
        f["sentence_len_std"]      = np.std(sent_lens)  if sent_lens else 0
        tl = text.lower()
        f["hedging_density"]    = sum(tl.count(w) for w in HEDGING_WORDS)    / len(words) if words else 0
        f["certainty_density"]  = sum(tl.count(w) for w in CERTAINTY_WORDS)  / len(words) if words else 0
        f["connector_density"]  = sum(tl.count(w) for w in CONNECTOR_WORDS)  / len(words) if words else 0
        contractions = ["n't","'ll","'re","'ve","'d","'m"]
        f["contraction_ratio"]  = sum(text.count(c) for c in contractions) / len(words) if words else 0

        # ─── NEW: AI PHRASE DENSITY ──────────────────────────────
        f["ai_phrase_density"]  = sum(tl.count(p) for p in AI_HEDGE_PHRASES) / max(len(sentences), 1)

        # ─── NEW: FUNCTION WORD FEATURES ────────────────────────
        fw_count = sum(1 for w in words if w.lower() in FUNCTION_WORDS)
        f["function_word_ratio"]  = fw_count / len(words) if words else 0
        top_fw = ["the","a","of","in","and","to","is","that","it","as"]
        for fw in top_fw:
            f[f"fw_{fw}"] = tl.count(f" {fw} ") / len(words) if words else 0

        # ─── NEW: PUNCTUATION ENTROPY ────────────────────────────
        punct_chars = [c for c in text if not c.isalnum() and not c.isspace()]
        if punct_chars:
            pc     = Counter(punct_chars)
            pp     = np.array(list(pc.values())) / len(punct_chars)
            f["punctuation_entropy"] = scipy_stats.entropy(pp)
        else:
            f["punctuation_entropy"] = 0.0

        # ─── NEW: READABILITY INDICES ────────────────────────────
        try:
            f["flesch_reading_ease"]   = textstat.flesch_reading_ease(text)
            f["flesch_kincaid_grade"]  = textstat.flesch_kincaid_grade(text)
            f["gunning_fog"]           = textstat.gunning_fog(text)
            f["smog_index"]            = textstat.smog_index(text)
            f["automated_readability"] = textstat.automated_readability_index(text)
            f["coleman_liau"]          = textstat.coleman_liau_index(text)
        except Exception:
            for k in ["flesch_reading_ease","flesch_kincaid_grade","gunning_fog",
                      "smog_index","automated_readability","coleman_liau"]:
                f[k] = np.nan

        # ─── NEW: POS TAG DISTRIBUTION (spaCy) ─────────────────
        doc  = nlp(text[:5000])   # cap for speed
        tags = [t.pos_ for t in doc]
        total_tags = len(tags) or 1
        for pos_tag in ["NOUN","VERB","ADJ","ADV","DET","ADP","PUNCT","PROPN","PRON","NUM"]:
            f[f"pos_{pos_tag.lower()}"] = tags.count(pos_tag) / total_tags

        # ─── NEW: DEPENDENCY TREE DEPTH ─────────────────────────
        depths = []
        for sent in doc.sents:
            root = [t for t in sent if t.head == t]
            if root:
                depths.append(self._tree_depth(root[0]))
        f["dep_depth_mean"] = float(np.mean(depths)) if depths else 0
        f["dep_depth_max"]  = float(np.max(depths))  if depths else 0

        # ─── NEW: SENTENCE-LEVEL PERPLEXITY (mean + variance) ────
        if self.use_sentence_ppl and sentences:
            ppls = []
            for sent in sentences[:15]:  # cap for speed
                p = sentence_perplexity(sent)
                if not np.isnan(p) and p < 1e6:
                    ppls.append(p)
            f["sent_ppl_mean"] = float(np.mean(ppls)) if ppls else np.nan
            f["sent_ppl_var"]  = float(np.var(ppls))  if ppls else np.nan
            f["sent_ppl_std"]  = float(np.std(ppls))  if ppls else np.nan
            # Low variance = suspiciously uniform (AI signal)
            f["sent_ppl_cv"]   = (f["sent_ppl_std"] / f["sent_ppl_mean"]
                                  if f["sent_ppl_mean"] and f["sent_ppl_mean"] > 0
                                  else np.nan)
        else:
            for k in ["sent_ppl_mean","sent_ppl_var","sent_ppl_std","sent_ppl_cv"]:
                f[k] = np.nan

        # ─── BURSTINESS ─────────────────────────────────────────
        f["burstiness"] = self._burstiness(words)

        return f

    def _tree_depth(self, token, depth=0):
        children = list(token.children)
        if not children:
            return depth
        return max(self._tree_depth(c, depth+1) for c in children)

    def _burstiness(self, words):
        if len(words) < 2:
            return 0
        pos_map = {}
        for i, w in enumerate(words):
            pos_map.setdefault(w, []).append(i)
        vars_ = []
        for positions in pos_map.values():
            if len(positions) > 1:
                vars_.append(np.var(np.diff(positions)))
        return float(np.mean(vars_)) if vars_ else 0


# ── Cell 5: Load Data ──────────────────────────────────────────
def encode_labels(series):
    return (series == "llm").astype(int)

hc3_train  = pd.read_csv("hc3_train.csv")
hc3_test   = pd.read_csv("hc3_test.csv")
eli5_train = pd.read_csv("eli5_train.csv")
eli5_test  = pd.read_csv("eli5_test.csv")

for df in [hc3_train, hc3_test, eli5_train, eli5_test]:
    df["label_encoded"] = encode_labels(df["label"])

print(f"HC3  Train={len(hc3_train):,} Test={len(hc3_test):,}")
print(f"ELI5 Train={len(eli5_train):,} Test={len(eli5_test):,}")

# ── Cell 6: Feature Extraction ─────────────────────────────────
extractor = FullStylometricExtractor(use_sentence_ppl=True)

print("\nExtracting features — HC3 train ...")
hc3_tr_feat  = extractor.extract_batch(hc3_train["text"].tolist())
print("Extracting features — HC3 test  ...")
hc3_te_feat  = extractor.extract_batch(hc3_test["text"].tolist())
print("Extracting features — ELI5 train ...")
eli5_tr_feat = extractor.extract_batch(eli5_train["text"].tolist())
print("Extracting features — ELI5 test  ...")
eli5_te_feat = extractor.extract_batch(eli5_test["text"].tolist())

print(f"\n✅ Feature vector dimensionality: {len(extractor.feature_names_)}")

# Impute NaNs with column medians (computed on train)
def impute_median(train_df, test_dfs):
    medians = train_df.median()
    train_df = train_df.fillna(medians)
    test_out = [df.fillna(medians) for df in test_dfs]
    return train_df, test_out, medians

# ── HC3-trained imputation ─────────────────────────────────────
hc3_medians        = hc3_tr_feat.median()
hc3_tr_feat        = hc3_tr_feat.fillna(hc3_medians)
hc3_te_feat        = hc3_te_feat.fillna(hc3_medians)
eli5_te_feat_hc3   = eli5_te_feat.fillna(hc3_medians)   # ELI5 test imputed with HC3 medians

# ── ELI5-trained imputation ────────────────────────────────────
eli5_medians       = eli5_tr_feat.median()
eli5_tr_feat       = eli5_tr_feat.fillna(eli5_medians)
eli5_te_feat       = eli5_te_feat.fillna(eli5_medians)
hc3_te_feat_eli5   = hc3_te_feat.fillna(eli5_medians)   # HC3 test imputed with ELI5 medians

# ── Cell 7: Classifiers ────────────────────────────────────────
CLASSIFIERS = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, C=1.0, solver="lbfgs",
        class_weight="balanced", random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1),

    "XGBoost": xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=1, use_label_encoder=False,
        eval_metric="logloss", random_state=42,
        n_jobs=-1, verbosity=0),
}

scaler = StandardScaler()

# ── Cell 8: Training & Evaluation ────────────────────────────
all_results = {}

train_configs = [
    ("HC3",  hc3_tr_feat,  hc3_train["label_encoded"],
     {"hc3_to_hc3":  (hc3_te_feat,    hc3_test["label_encoded"]),
      "hc3_to_eli5": (eli5_te_feat_hc3, eli5_test["label_encoded"])}),

    ("ELI5", eli5_tr_feat, eli5_train["label_encoded"],
     {"eli5_to_eli5": (eli5_te_feat,   eli5_test["label_encoded"]),
      "eli5_to_hc3":  (hc3_te_feat_eli5, hc3_test["label_encoded"])}),
]

for clf_name, clf in CLASSIFIERS.items():
    print(f"\n{'='*60}")
    print(f"Classifier: {clf_name}")
    print(f"{'='*60}")
    all_results[clf_name] = {}

    for ds_name, X_tr_raw, y_tr, test_sets in train_configs:
        X_tr = scaler.fit_transform(X_tr_raw)
        clf.fit(X_tr, y_tr)

        for eval_name, (X_te_raw, y_te) in test_sets.items():
            X_te = scaler.transform(X_te_raw)
            probs = clf.predict_proba(X_te)[:, 1]
            preds = (probs > 0.5).astype(int)

            auc    = roc_auc_score(y_te, probs)
            brier  = brier_score_loss(y_te, probs)
            ll     = log_loss(y_te, probs)
            acc    = accuracy_score(y_te, preds)

            all_results[clf_name][eval_name] = {
                "y_true":               np.array(y_te),
                "y_pred":               preds,
                "detectability_scores": probs,
                "roc_auc":              auc,
                "brier_score":          brier,
                "log_loss":             ll,
                "accuracy":             acc,
            }
            print(f"  {eval_name:20s} → AUC={auc:.4f}  Acc={acc:.4f}")

# ── Cell 9: Feature Importance (SHAP) ─────────────────────────
print("\n" + "="*70)
print("SHAP FEATURE IMPORTANCE — XGBoost on HC3")
print("="*70)

# Refit XGBoost with scaled HC3 training data for SHAP
X_tr_scaled  = scaler.fit_transform(hc3_tr_feat)
xgb_model    = CLASSIFIERS["XGBoost"]
xgb_model.fit(X_tr_scaled, hc3_train["label_encoded"])

explainer    = shap.TreeExplainer(xgb_model)
X_te_scaled  = scaler.transform(hc3_te_feat)
shap_values  = explainer.shap_values(X_te_scaled[:500])

feat_names = extractor.feature_names_

# Summary plot
plt.figure(figsize=(10, 12))
shap.summary_plot(
    shap_values, X_te_scaled[:500],
    feature_names=feat_names,
    max_display=30,
    show=False,
    plot_type="bar",
)
plt.title("XGBoost SHAP Feature Importances (HC3 Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/shap_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

# Detailed beeswarm
plt.figure(figsize=(10, 14))
shap.summary_plot(
    shap_values, X_te_scaled[:500],
    feature_names=feat_names,
    max_display=30,
    show=False,
)
plt.title("SHAP Beeswarm — Direction & Magnitude of Each Feature", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 10: Random Forest Feature Importance ─────────────────
rf_model = CLASSIFIERS["Random Forest"]
rf_model.fit(scaler.fit_transform(hc3_tr_feat), hc3_train["label_encoded"])

importances = pd.Series(rf_model.feature_importances_, index=feat_names)
top30 = importances.nlargest(30)

plt.figure(figsize=(9, 10))
top30.sort_values().plot(kind="barh", color="#3498db")
plt.xlabel("Gini Importance")
plt.title("Random Forest: Top 30 Feature Importances (HC3)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/rf_feature_importance.png", dpi=150)
plt.show()

# ── Cell 11: Feature Distributions by Label ───────────────────
KEY_FEATURES = [
    "sent_ppl_cv", "connector_density", "ai_phrase_density",
    "flesch_kincaid_grade", "dep_depth_mean", "pos_adj",
    "type_token_ratio", "hedging_density", "function_word_ratio",
    "punctuation_entropy",
]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle("Key Feature Distributions: Human vs LLM (HC3 Test Set)",
             fontsize=14, fontweight="bold")

for ax, feat in zip(axes.flatten(), KEY_FEATURES):
    if feat in hc3_te_feat.columns:
        vals = hc3_te_feat[feat].fillna(hc3_te_feat[feat].median())
        human_vals = vals[hc3_test["label_encoded"]==0]
        llm_vals   = vals[hc3_test["label_encoded"]==1]
        clip_q = (vals.quantile(0.01), vals.quantile(0.99))

        ax.hist(human_vals.clip(*clip_q), bins=30, alpha=0.6,
                label="Human", color="#3498db", density=True)
        ax.hist(llm_vals.clip(*clip_q),   bins=30, alpha=0.6,
                label="LLM",   color="#e74c3c", density=True)
        ax.set_title(feat, fontsize=9, fontweight="bold")
        ax.set_xlabel("Value"); ax.legend(fontsize=7); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Cell 12: Score Distributions ──────────────────────────────
for clf_name, results in all_results.items():
    eval_names = list(results.keys())
    n = len(eval_names)
    if n == 0:
        continue

    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle(f"{clf_name}: Detectability Score Distributions",
                 fontsize=13, fontweight="bold")

    for ax, eval_name in zip(axes, eval_names):
        d = results[eval_name]
        ax.hist(d["detectability_scores"][d["y_true"]==0], bins=30, alpha=0.6,
                label="Human", color="#3498db", range=(0,1))
        ax.hist(d["detectability_scores"][d["y_true"]==1], bins=30, alpha=0.6,
                label="LLM",   color="#e74c3c", range=(0,1))
        ax.set_title(f"{eval_name}\nAUC={d['roc_auc']:.3f}", fontsize=10)
        ax.axvline(0.5, color="k", linestyle="--", alpha=0.4)
        ax.legend(); ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/{clf_name.replace(' ','_')}_distributions.png",
                dpi=150, bbox_inches="tight")
    plt.show()

# ── Cell 13: Cross-Domain Feature Stability ───────────────────
print("\n" + "="*70)
print("CROSS-DOMAIN FEATURE STABILITY ANALYSIS")
print("="*70)

# Compare feature means: human vs LLM across HC3 and ELI5
for ds_name, feat_df, label_series in [
    ("HC3",  hc3_te_feat,  hc3_test["label_encoded"]),
    ("ELI5", eli5_te_feat, eli5_test["label_encoded"]),
]:
    human = feat_df[label_series==0]
    llm   = feat_df[label_series==1]
    diff  = (llm.mean() - human.mean()).abs()
    top10 = diff.nlargest(10)
    print(f"\n{ds_name} — Top 10 features by |LLM mean − Human mean|:")
    for feat, val in top10.items():
        print(f"  {feat:40s}  Δ={val:.4f}")

# ── Cell 14: Performance Summary ──────────────────────────────
rows = []
for clf_name, results in all_results.items():
    for eval_name, d in results.items():
        rows.append({
            "Classifier":      clf_name,
            "Evaluation":      eval_name,
            "ROC-AUC":         d["roc_auc"],
            "Accuracy":        d["accuracy"],
            "Brier Score":     d["brier_score"],
            "Log Loss":        d["log_loss"],
            "Mean Human Score":d["detectability_scores"][d["y_true"]==0].mean(),
            "Mean LLM Score":  d["detectability_scores"][d["y_true"]==1].mean(),
            "Score Separation":(d["detectability_scores"][d["y_true"]==1].mean() -
                                d["detectability_scores"][d["y_true"]==0].mean()),
        })

summary_df = pd.DataFrame(rows)
print("\n" + "="*70)
print("STYLOMETRIC HYBRID PERFORMANCE SUMMARY")
print("="*70)
print(summary_df.round(4).to_string(index=False))
summary_df.to_csv(f"{RESULTS_DIR}/stylometric_performance_summary.csv", index=False)

# Heatmap: AUC by Classifier × Evaluation
pivot = summary_df.pivot_table(index="Classifier", columns="Evaluation", values="ROC-AUC")
plt.figure(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=0.5)
plt.title("Stylometric Hybrid: AUC Heatmap", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/auc_heatmap.png", dpi=150)
plt.show()

# ── Cell 15: Save All Artefacts ────────────────────────────────
with open(f"{RESULTS_DIR}/stylometric_results.pkl", "wb") as f:
    pickle.dump(all_results, f)

# Save feature matrix for interpretability analysis in other notebooks
hc3_te_feat["label"] = hc3_test["label"].values
hc3_te_feat.to_csv(f"{RESULTS_DIR}/hc3_feature_matrix.csv", index=False)

eli5_te_feat["label"] = eli5_test["label"].values
eli5_te_feat.to_csv(f"{RESULTS_DIR}/eli5_feature_matrix.csv", index=False)

print(f"\n✅ All results saved to {RESULTS_DIR}/")
print("🎯 Stylometric and Statistical Hybrid Detector complete.")
print("\n📋 Feature count breakdown:")
all_feats = extractor.feature_names_
stage2a_feats = [f for f in all_feats if f in [
    "word_count","char_count","sentence_count","avg_word_len","avg_sentence_len",
    "type_token_ratio","hapax_ratio","comma_density","period_density",
    "bigram_repetition","trigram_repetition","word_entropy",
    "hedging_density","certainty_density","connector_density",
    "contraction_ratio","burstiness","sentence_len_variance","sentence_len_std",
    "question_mark_ratio","exclamation_ratio"
]]
new_feats = [f for f in all_feats if f not in stage2a_feats]
print(f"  Stage 2A carried over : {len(stage2a_feats)} features")
print(f"  New features added    : {len(new_feats)} features")
print(f"  Total                 : {len(all_feats)} features")